# KosenMap Website 取扱説明書 —— 目次

**この Website(公開ページ・管理画面・本番ホスト)を動かすための取扱説明書。**
読むだけでなく、**書いてあるコマンドをそのセルから実行できる。**

対象は本番 `ito4.jp`(`/opt/kosenmap`)と、この PC の `server/scripts`。
**繋ぐ利用者は 2 つ**(2026-09-18 に分けた。[12](12-hardening-2026-09-15.ipynb) §7-4 B): 配備・`%%host`・控えは **`kmops`**(鍵 `~\.ssh\km_ops`。docker あり・**sudo なし**)、`sudo` が要るセルは **`km`**(鍵 `~\.ssh\km_vps`。**docker なし**)。
Android アプリは別リポジトリなので、ここには入っていない。

| ノートブック | 何が書いてあるか |
|---|---|
| [00-start](00-start.ipynb) | **この取扱説明書の使い方。** 準備と、セルの型 |
| [01-daily-check](01-daily-check.ipynb) | **日々の確認。** 自己検査・ホストの様子・メールが届いているか |
| [02-deploy](02-deploy.ipynb) | **配備。** 下見・配備・後片付け・困ったとき |
| [03-backup](03-backup.ipynb) | **バックアップと復元。** 取る・開く・週次タスク・添付の復号・戻す |
| [04-host-jobs-and-mail](04-host-jobs-and-mail.ipynb) | **ホストの定期処理とメール。** cron の時刻表・届くメールの一覧・試しに送る |
| [05-containers](05-containers.ipynb) | **コンテナの更新と追加。** Logto の上げ方・固定・足し方 |
| [06-emergency](06-emergency.ipynb) | **もしものとき。** まず叩く1本と、症状別の見どころ |
| [07-map-qr](07-map-qr.ipynb) | **地図の配信と QR。** |
| [08-architecture](08-architecture.ipynb) | **どう出来ているか。** 構造・設定の読み方・落とし穴・設計の約束 |
| [09-new-host](09-new-host.ipynb) | **新しいホストを作る・移す。** VPS の構築から切り替えまで |
| [10-security-review-2026-09-14](10-security-review-2026-09-14.ipynb) | **診断の報告書(2026-09-14)。** 何を見つけて何を直したか・本番で実行する手順 |
| [11-getting-started](11-getting-started.ipynb) | **新しく使う人の入口。** 全体の図・部品の役割・管理画面・Android アプリ・用語集 |
| [12-hardening-2026-09-15](12-hardening-2026-09-15.ipynb) | **多層防御の底上げ(2026-09-15)。** 網の分割・read_only・Soketi の Node 24・Logto の DB 利用者・管理画面の別オリジン・本番への当て方 |
| [13-local-env](13-local-env.ipynb) | **ローカル環境(LAN の検証機)。** `.env` の `KM_ENV=local` で、Let's Encrypt を使わずに本番と同じ構成を立てる |
| [14-domain-ito4](14-domain-ito4.ipynb) | **ito4.jp に統一する(2026-09-17)。** 旧 ito8795.com は同時に手放す・audience も https://ito4.jp/api へ・管理画面は admin.ito4.jp・外部スキャンの指摘 |
| [15-staff-org](15-staff-org.ipynb) | **教職員の自動付与(2026-09-18〜)。** 学校ドメインの JIT で Logto の組織へ入れ、組織ロールで「地図の錠を通る・教職員氏名を見る・閲覧不可の地点を見る」を与える |
| [17-cloudflare](17-cloudflare.ipynb) | Cloudflare に載せるときの計画(書くだけ・当てていない) |
| [18-room-shape-research](18-room-shape-research.ipynb) | 写真から部屋の形を作る(調査の報告) |

記録として残している文書:

| 文書 | 中身 |
|---|---|
| [status.md](status.md) | **いまどうなっているか。** 実測値・踏んだ罠・残作業 |
| [plan.md](plan.md) | **これから何をするか。** 優先順と、やらないと決めたこと |
| [../Old/docs/](../Old/docs/) | 以前の手順書(md)。**細部の経緯はここ**。ノートブックに移した内容の元 |
| [17-cloudflare](17-cloudflare.ipynb) | Cloudflare に載せるときの計画(書くだけ・当てていない) |
| [18-room-shape-research](18-room-shape-research.ipynb) | 写真から部屋の形を作る(調査の報告) |

In [ ]:
# 最初に1回だけ実行する(%%ps / %%host / %%terminal が使えるようになる)
import sys, pathlib
for _d in (pathlib.Path.cwd(), pathlib.Path.cwd() / 'docs', pathlib.Path.cwd() / 'server' / 'docs'):
    if (_d / 'km_nb.py').exists():
        sys.path.insert(0, str(_d))
        break
import km_nb
km_nb.load()


# 17. Cloudflare に載せるときの計画(2026-10-06。**書くだけ・当てていない**)

利用者の質問「Cloudflare にドメインを接続しようと考えているがどうか」への答えと、載せるときの手順・戻し方。
**今は何も変えていない。** 決めるのは利用者。

## 0. 結論(先に)

- **公開ページ(443)は載せてよい。** 帯域と攻撃の肩代わり・国の拒否(WAF)をサーバーの資源を使わずに得られる
- **ただし今の作りのままでは載せられない口がある。** 3001(Logto)・6001(Soketi)・8281(phpMyAdmin)・3002(Logto の Console)は
  Cloudflare の中継が通すポートに無い(中継できるのは HTTP 80/8080/8880/2052/2082/2086/2095、HTTPS 443/2053/2083/2087/2096/8443)
- **無料の版は 1 回の本文が 100MB まで。** APK の差し替え(`/admin/downloads.php`、最大 210MB)は中継の外に置く必要がある
- 進め方は「名前ごとに少しずつ橙(中継)にする」。**いつでも灰(DNS だけ)に戻せば元どおり**

> 上の数字(ポート・100MB)は Cloudflare の公開資料に基づく。**当てる前に最新の資料で確かめること**(この文書を書いた時点で読み直していない)。

## 1. 名前の分け方(案)

| 今 | 載せたあと(案) | 中継 | 理由 |
|---|---|---|---|
| `ito4.jp:443`(公開ページ・アプリの API) | 同じ | **橙** | 本体。帯域と攻撃を肩代わりさせる |
| `ito4.jp:3001`(Logto。サインイン) | `auth.ito4.jp:443` | **橙** | 3001 は中継できない。443 の別名にする |
| `ito4.jp:6001`(Soketi。WebSocket) | `ws.ito4.jp:443` | 橙 | WebSocket は 443 で中継できる |
| `admin.ito4.jp`(管理画面・8281・3002) | 同じ | **灰** | IP で絞っている(allow-admin・タスクマネージャー)。APK の 210MB もここ |
| メール(MX・mail の A) | 同じ | **灰(必ず)** | メールは中継しない |

- **灰の名前が残る限り、サーバーの IP は見える。** 公開ページを橙にしても、IP を直に叩かれる道は残る
  → 80/443 は **Cloudflare の IP からだけ**受ける(nftables か nginx の allow)。管理用の名前は今までどおり IP で絞る
- 名前を変える口(Logto・Soketi)は**アプリの作り直し**が要る(`LOGTO_ENDPOINT`・WebSocket の接続先は APK に入っている)。
  古い APK が残るあいだは 3001・6001 も灰で残す(移行期間)

## 2. サーバーの側で直すもの

| 何を | なぜ | どこ |
|---|---|---|
| **本当の送り元を取る**(`set_real_ip_from` に Cloudflare の範囲・`real_ip_header CF-Connecting-IP`) | 直さないと、回数の制限・IP の許可・訪問者の記録・ログが**全部 Cloudflare の IP** を見る。回数の制限は全員まとめて 1 人扱いになる | `nginx/default.conf.template` の http(範囲は定期的に取り直す) |
| **国の拒否を Cloudflare の WAF へ移す** | 中継を通ると、ホストの nftables(`host-geoblock.sh`)からは送り元が Cloudflare に見える。灰の名前(管理用)にはホストの拒否がそのまま効く | Cloudflare の画面(Security → WAF → 国) |
| **証明書** | 橙の名前は Cloudflare が利用者に証明書を出す。サーバーとの間は「Full (strict)」にし、サーバーには今の Let's Encrypt か Cloudflare の Origin 証明書 | `scripts/host-cert.sh`。HTTP-01 は `/.well-known/acme-challenge/` を中継が素通しすれば今のまま動く(「常に HTTPS」で 80 を曲げないこと) |
| **CAA** | 橙の名前の証明書は Cloudflare の CA が出す。CAA を書いているなら、その CA も許す | DNS(Cloudflare は自分の CA を足すことがある。当てる前に確かめる) |
| **キャッシュ** | 既定では HTML と PHP の応答はキャッシュしないが、`Cache-Control` の無い静的ファイルは溜まる。地図の資材は `?v=` 付きなので害は無い | 確かめるだけ |
| **Logto の ENDPOINT** | 名前を `auth.ito4.jp` にするなら、Logto の ENDPOINT・アプリの戻り先(redirect URI)・web の `LOGTO_ENDPOINT` を揃えて変える | `.env`・Logto の Console・Android の build 設定 |

## 3. 手順(案)

1. Cloudflare に `ito4.jp` を足し、**今の DNS をそのまま取り込む。全部「灰」**のまま
2. 登録業者で**ネームサーバーを Cloudflare に替える**(ここまでは中身が何も変わらない)
3. `dig` で全部の記録が同じ値を返すことを確かめる(MX・TXT・CAA も)
4. サーバー: `set_real_ip_from`(Cloudflare の範囲)を入れて配備する(灰のうちは効かないだけで害は無い)
5. `ito4.jp` を**橙**にする。確かめること:
   - 公開ページ・アプリの地図の取得
   - 訪問者の記録に本当の IP が出る
   - 回数の制限が 1 人ずつ効く
6. 80/443 を Cloudflare の範囲からだけ受けるようにする(灰の名前・メールは対象外)
7. (別の段)Logto と Soketi を 443 の別名へ移す。**アプリの新しい版を配ってから**旧ポートを閉じる

## 4. 戻し方

- 名前ごとに**灰へ戻す**だけで、DNS の答えはサーバーの IP に戻る(数分で効く)
- ネームサーバーを元の業者へ戻すのは最後の手段(反映に最大 48 時間ほどかかりうる)
- 6 の「Cloudflare の範囲からだけ受ける」を入れていたら、**先にそれを外す**(灰に戻した途端に全員が弾かれる)

## 5. 未確認のこと(当てる前に)

- 中継できるポートと本文の上限(無料の版)の最新の値
- CAA を Cloudflare が自動で足すかどうか
- 学校のネットワークから Cloudflare の名前に届くか(フィルタ)
- Android の古い版が、3001・6001 をどのくらいの期間使い続けるか(自動アップデートの広がり)